# Tiffany Video Sorter — Drive Edition (Claude subscription via OAuth token)

Sorts raw video clips in Google Drive into topic / A-roll / B-roll folders without downloading anything to your computer.

**How it works per clip:**
1. Streams the video from Drive (Google-to-Google, fast)
2. Extracts audio (a few MB) — discards video bytes
3. Whisper transcribes on the free Colab T4 GPU
4. Claude Haiku 4.5 (via Anthropic SDK, signed in with your subscription token) classifies the transcript
5. Drive API moves the file into the right folder (metadata-only — instant, no upload)
6. File is renamed; original filename is preserved at the end (e.g. `ai-tools-a-roll-03__C2004.mp4`)

After all clips: a single Google Doc named **All Transcripts** is created containing every transcript.

**Cost:** $0 marginal — uses your existing Claude subscription quota via OAuth token (no API credits charged). Whisper is local. Drive moves are metadata.

**Safety:** Never deletes any file. Re-running is idempotent (state cache on Drive).

## Step 1 — Set the runtime to GPU

Runtime → Change runtime type → **T4 GPU** → Save. (Free tier includes this.)

In [ ]:
# Step 2 — install dependencies (Whisper, ffmpeg, anthropic SDK).
# No Node.js, no Claude Code CLI. Uses the Anthropic Python SDK directly.

import subprocess, sys, shutil

def sh(cmd, check=True):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout: print(r.stdout)
    if r.stderr: print(r.stderr, file=sys.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f'command failed: {cmd}')
    return r

# ffmpeg
if shutil.which('ffmpeg') is None:
    sh('apt-get update -qq && apt-get install -y -qq ffmpeg')
print('ffmpeg:', shutil.which('ffmpeg'))

# Python deps
sh('pip install -q --upgrade openai-whisper anthropic google-api-python-client google-auth google-auth-httplib2 google-auth-oauthlib')

import anthropic, whisper
print('anthropic:', anthropic.__version__)
print('whisper imported OK')
print('ALL DEPENDENCIES OK ✓')

## Step 3 — Sign in with your Claude subscription (OAuth token)

We generate a long-lived OAuth token on your Mac (one command), then paste it here. This bills against your Claude subscription — $0 marginal cost — instead of charging the Anthropic API.

**On your Mac terminal, run:**

```
claude setup-token
```

It opens a browser, you click Authorize, and prints a token starting with `sk-ant-oat-...`. Copy that token, paste into the cell below.

In [ ]:
# Step 3a — set your Claude subscription OAuth token.
# Generate it on your Mac:  claude setup-token
# Paste the printed sk-ant-oat-... token below, then run this cell.

import os
CLAUDE_OAUTH_TOKEN = 'PASTE_YOUR_sk-ant-oat01_TOKEN_HERE'  # starts with sk-ant-oat-...

assert CLAUDE_OAUTH_TOKEN.startswith('sk-ant-oat'), \
    'Expected an OAuth token starting with sk-ant-oat-. Run  claude setup-token  on your Mac.'

# The Anthropic SDK accepts the OAuth token as auth_token (Bearer header).
# We stash it in env so the rest of the notebook can rebuild the client easily.
os.environ['CLAUDE_CODE_OAUTH_TOKEN'] = CLAUDE_OAUTH_TOKEN
# Clear any API key so the SDK does not prefer it.
os.environ.pop('ANTHROPIC_API_KEY', None)

import anthropic
client = anthropic.Anthropic(auth_token=CLAUDE_OAUTH_TOKEN)

resp = client.messages.create(
    model='claude-haiku-4-5',
    max_tokens=20,
    messages=[{'role': 'user', 'content': 'Reply with the single word: ready'}],
)
print('Subscription auth check:', resp.content[0].text.strip())
print('OAuth token OK ✓')

In [ ]:
# Step 3b — (no longer needed; Step 3a already verified the token)
print('Skipped: Step 3a already verified the OAuth token.')

## Step 4 — Configuration

- `RAW_FOLDER_URL`: open the Drive folder containing your raw clips in a browser and paste its URL.
- The output library and transcripts Doc are created next to the raw folder (same parent).

In [ ]:
RAW_FOLDER_URL = 'PASTE_DRIVE_FOLDER_URL_HERE'
OUTPUT_LIBRARY_NAME = 'Sorted Library'
TRANSCRIPT_DOC_NAME = 'All Transcripts'

# ---- Smoke-test mode --------------------------------------------------------
# Set TEST_LIMIT to a small number (e.g. 3) to process only the first N pending
# clips. Inspect the result in Drive, then set TEST_LIMIT = None to run the rest.
TEST_LIMIT = 3
# -----------------------------------------------------------------------------

BATCH_SIZE = 15
WHISPER_MODEL = 'medium'    # tiny / base / small / medium / large-v3
LANGUAGE = 'en'
SEED_TOPICS = ['e-commerce', 'ai-tools', 'business-automation']
AUTO_THRESHOLD = 0.85
REVIEW_THRESHOLD = 0.60
CLAUDE_MODEL = 'claude-haiku-4-5-20251001'  # cheapest capable; matcher only

import os, re
m = re.search(r'/folders/([a-zA-Z0-9_-]+)', RAW_FOLDER_URL)
assert m, 'RAW_FOLDER_URL must look like https://drive.google.com/drive/folders/...'
RAW_FOLDER_ID = m.group(1)
print('Raw folder ID:', RAW_FOLDER_ID)
print('TEST_LIMIT:', TEST_LIMIT if TEST_LIMIT is not None else 'OFF (process all pending)')


In [ ]:
# Step 5 — authenticate Google Drive + Docs
# On the popup, MAKE SURE BOTH "Google Drive" AND "Google Docs" scopes are granted.
# If you accidentally denied one, run:
#     from google.colab import auth; auth.authenticate_user(force_remount=True)
# in a fresh cell and accept both.

from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

auth.authenticate_user()
drive = build('drive', 'v3', cache_discovery=False)
docs  = build('docs',  'v1', cache_discovery=False)

# Fail-fast smoke test: list one Drive file. (No write/delete — respecting the
# hardcore "never delete anything" rule. Docs scope will be verified at Step 13
# when we actually write the transcripts doc.)
try:
    drive.files().list(pageSize=1, fields='files(id,name)').execute()
    print('Drive API OK')
except HttpError as e:
    raise RuntimeError(
        'Drive API failed. Re-run this cell and grant Drive access on the popup.\n' + str(e)
    )

print('\nGoogle auth OK ✓  (Docs scope is checked later, in Step 13.)')

In [ ]:
# Step 6 — helpers (list, ensure folder, move, audio extraction) — hardened
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
from googleapiclient.errors import HttpError
import io, json, subprocess, time, random
from pathlib import Path

VIDEO_MIME_PREFIXES = ('video/',)
VIDEO_EXTS = ('.mp4','.mov','.m4v','.mkv','.avi','.webm')
WORKDIR = Path('/content/work'); WORKDIR.mkdir(exist_ok=True)
AUDIO_DIR = WORKDIR / 'audio'; AUDIO_DIR.mkdir(exist_ok=True)

def drive_retry(fn, *, tries=5, base_delay=2):
    """Exponential-backoff wrapper for transient Drive API errors (429/5xx)."""
    for i in range(tries):
        try:
            return fn()
        except HttpError as e:
            code = getattr(e.resp, 'status', 0)
            if code in (429, 500, 502, 503, 504) and i < tries - 1:
                sleep = base_delay * (2 ** i) + random.uniform(0, 1)
                print(f'    drive retry in {sleep:.1f}s (HTTP {code})')
                time.sleep(sleep); continue
            raise

def _q_escape(s):
    """Escape a string for use inside a Drive `q` parameter."""
    return s.replace('\\', '\\\\').replace("'", "\\'")

def list_videos(folder_id):
    files, token = [], None
    while True:
        resp = drive_retry(lambda: drive.files().list(
            q=f"'{folder_id}' in parents and trashed=false",
            fields='nextPageToken, files(id,name,mimeType,size,parents)',
            pageSize=200, pageToken=token).execute())
        for f in resp.get('files', []):
            if (f.get('mimeType','').startswith(VIDEO_MIME_PREFIXES) or
                f['name'].lower().endswith(VIDEO_EXTS)):
                files.append(f)
        token = resp.get('nextPageToken')
        if not token: break
    return files

def get_parent(folder_id):
    return drive_retry(lambda: drive.files().get(
        fileId=folder_id, fields='parents,name').execute())

def ensure_folder(name, parent_id):
    q = (f"name='{_q_escape(name)}' and '{parent_id}' in parents and "
         "mimeType='application/vnd.google-apps.folder' and trashed=false")
    res = drive_retry(lambda: drive.files().list(
        q=q, fields='files(id,name)').execute()).get('files', [])
    if res: return res[0]['id']
    body = {'name': name, 'mimeType': 'application/vnd.google-apps.folder',
            'parents': [parent_id]}
    return drive_retry(lambda: drive.files().create(
        body=body, fields='id').execute())['id']

def ensure_path(parts, root_id):
    cur = root_id
    for p in parts:
        cur = ensure_folder(p, cur)
    return cur

def move_file(file_id, new_parent_id, new_name=None):
    """Re-parent (and optionally rename). NEVER deletes — only moves metadata.
    Respects the hardcore 'no deletion' rule: the file content is untouched."""
    cur = drive_retry(lambda: drive.files().get(
        fileId=file_id, fields='parents,name').execute())
    prev = ','.join(cur.get('parents', []))
    body = {'name': new_name} if new_name else {}
    return drive_retry(lambda: drive.files().update(
        fileId=file_id, addParents=new_parent_id, removeParents=prev,
        body=body, fields='id,parents,name').execute())

def _safe_unlink(p):
    """Delete a LOCAL Colab tmp file (not Drive). Required to keep Colab disk
    from filling up. User's Drive content is never touched."""
    try: Path(p).unlink(missing_ok=True)
    except Exception: pass

class NoAudioTrackError(Exception): pass

def download_audio_only(file_id, dest_audio_path):
    """Stream video from Drive into Colab and extract audio only (a few MB).
    Returns the audio path on success, raises NoAudioTrackError if the source
    video has no audio track."""
    req = drive.files().get_media(fileId=file_id)
    tmp_video = WORKDIR / f'_tmp_{file_id}.bin'
    try:
        fh = io.FileIO(tmp_video, 'wb')
        downloader = MediaIoBaseDownload(fh, req, chunksize=64 * 1024 * 1024)
        done = False
        while not done:
            for attempt in range(4):
                try:
                    _, done = downloader.next_chunk(); break
                except HttpError as e:
                    if attempt == 3: raise
                    time.sleep(2 ** attempt)
        fh.close()

        # Probe for an audio stream; bail early on silent clips.
        probe = subprocess.run(
            ['ffprobe','-v','error','-select_streams','a',
             '-show_entries','stream=codec_type','-of','csv=p=0', str(tmp_video)],
            capture_output=True, text=True)
        if 'audio' not in (probe.stdout or ''):
            raise NoAudioTrackError(f'{file_id}: no audio stream')

        r = subprocess.run(
            ['ffmpeg','-y','-loglevel','error','-i', str(tmp_video),
             '-vn','-ac','1','-ar','16000','-b:a','64k', str(dest_audio_path)],
            capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(f'ffmpeg failed: {r.stderr.strip()[:300]}')
        return dest_audio_path
    finally:
        _safe_unlink(tmp_video)  # always free local disk

print('helpers loaded')

In [ ]:
# Step 7 — discover/create the output structure in Drive
raw_meta = get_parent(RAW_FOLDER_ID)
PARENT_ID = raw_meta['parents'][0]
OUTPUT_ROOT = ensure_folder(OUTPUT_LIBRARY_NAME, PARENT_ID)
STATE_FOLDER = ensure_folder('_state', OUTPUT_ROOT)
REVIEW_UNMATCHED = ensure_path(['_review', 'unmatched'], OUTPUT_ROOT)
REVIEW_NOAUDIO = ensure_path(['_review', 'no-audio'], OUTPUT_ROOT)
print(f'Output library:  https://drive.google.com/drive/folders/{OUTPUT_ROOT}')
print(f'Raw folder:      {raw_meta["name"]} ({RAW_FOLDER_ID})')

In [ ]:
# Step 8 — list pending clips (skip already-processed via _state cache on Drive)
def list_state_cache():
    res = drive_retry(lambda: drive.files().list(
        q=f"'{STATE_FOLDER}' in parents and trashed=false",
        fields='files(id,name)', pageSize=1000).execute())
    return {f['name']: f['id'] for f in res.get('files', [])}

all_videos = list_videos(RAW_FOLDER_ID)
# Deterministic order so TEST_LIMIT is reproducible across runs.
all_videos.sort(key=lambda v: v['name'].lower())

cache = list_state_cache()
def state_name(v): return f"{v['id']}.json"

cached_n  = sum(1 for v in all_videos if state_name(v) in cache)
pending   = [v for v in all_videos if state_name(v) not in cache]

if TEST_LIMIT is not None:
    pending = pending[:TEST_LIMIT]
    print(f'TEST MODE: processing first {TEST_LIMIT} pending clips only.')

print(f'{len(all_videos)} videos in raw folder | {cached_n} already done '
      f'| {len(pending)} will be processed this run')

In [ ]:
# Step 9 — load Whisper (once). Auto-detects GPU; falls back to CPU with a warning.
import whisper, torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cpu':
    print('⚠️  No GPU detected. Transcription will be slow (CPU).')
    print('   Fix: Runtime → Change runtime type → T4 GPU → Save, then re-run.')
else:
    print(f'GPU: {torch.cuda.get_device_name(0)}')

print(f'Loading Whisper "{WHISPER_MODEL}" on {device} (first run downloads ~1.5GB)...')
wm = whisper.load_model(WHISPER_MODEL, device=device)
print('whisper ready')

In [ ]:
# Step 10 — transcribe + classify (Anthropic SDK with subscription OAuth token)
import math, statistics
import anthropic

CLAUDE_MODEL = 'claude-haiku-4-5'

# Rebuild a client from the token set in Step 3a so this cell is re-runnable.
_OAT = os.environ.get('CLAUDE_CODE_OAUTH_TOKEN', '')
assert _OAT.startswith('sk-ant-oat'), 'Run Step 3a first to set the OAuth token.'
_anthro = anthropic.Anthropic(auth_token=_OAT)

MATCHER_SYSTEM = (
    "You classify Tiffany Parra's YouTube clips. Channel focus: e-commerce, AI tools, business automation. "
    "Return ONLY a JSON object (no prose, no markdown fences) matching this schema: "
    "{\"matched_topic\": <slug or _unmatched or general>, \"confidence\": 0-1, "
    "\"shot_type\": \"a-roll|b-roll|talking-head\", \"tags\": [3-6 short phrases], "
    "\"reason\": <one sentence>}. "
    "Rules: a-roll = direct address with full sentences; talking-head = long monologue; "
    "b-roll = minimal/no speech. matched_topic 'general' = reusable across topics (CTAs, scenery). "
    "confidence >=0.85 = clearly on-topic; 0.60-0.84 = partial; <0.60 = _unmatched. "
    "Only use slugs from the provided topic list (or general / _unmatched)."
)

def word_confidence(segments):
    probs = []
    for s in segments:
        lp = s.get('avg_logprob')
        if lp is not None:
            probs.append(math.exp(lp))
    return round(statistics.mean(probs), 3) if probs else None

def transcribe_audio(audio_path):
    r = wm.transcribe(str(audio_path), language=LANGUAGE, word_timestamps=False, verbose=False)
    text = (r.get('text') or '').strip()
    segs = r.get('segments') or []
    conf = word_confidence(segs)
    has_speech = bool(text and len(text) >= 8 and (conf is None or conf >= 0.2))
    return {'transcript': text, 'confidence': conf, 'has_speech': has_speech,
            'language': r.get('language')}

_JSON_RE = re.compile(r'\{.*\}', re.DOTALL)

def claude_classify(transcript_text, topic_list):
    """Call the Anthropic SDK with the subscription token. Retries once on transient failure."""
    user_msg = (
        f"Topics available: {json.dumps(topic_list)}\n\n"
        f"Transcript:\n{transcript_text}\n\n"
        f"Output the JSON object now (no markdown, no commentary)."
    )
    for attempt in range(2):
        try:
            resp = _anthro.messages.create(
                model=CLAUDE_MODEL,
                max_tokens=600,
                system=MATCHER_SYSTEM,
                messages=[{'role': 'user', 'content': user_msg}],
            )
            raw = ''.join(b.text for b in resp.content if hasattr(b, 'text')).strip()
            m = _JSON_RE.search(raw)
            if not m:
                raise RuntimeError(f'no JSON in API output: {raw[:300]}')
            return json.loads(m.group(0))
        except (anthropic.APIError, anthropic.APIConnectionError, RuntimeError, json.JSONDecodeError) as e:
            if attempt == 0:
                print(f'    classify retry: {type(e).__name__}: {str(e)[:120]}')
                time.sleep(3)
                continue
            raise

def classify(transcript_text, has_speech, topic_list):
    if not has_speech:
        return {'matched_topic': 'general', 'confidence': 0.0, 'shot_type': 'b-roll',
                'tags': [], 'reason': 'no speech detected'}
    return claude_classify(transcript_text, topic_list)

print('classifier ready (Anthropic SDK + subscription OAuth, model=' + CLAUDE_MODEL + ')')

In [ ]:
# Step 11 — load or seed topics.json (lives in Drive _state)
TOPICS_NAME = 'topics.json'

def read_state_json(name):
    files = drive.files().list(q=f"'{STATE_FOLDER}' in parents and name='{name}' and trashed=false",
                               fields='files(id)').execute().get('files', [])
    if not files: return None
    buf=io.BytesIO()
    dl=MediaIoBaseDownload(buf, drive.files().get_media(fileId=files[0]['id']))
    done=False
    while not done: _, done = dl.next_chunk()
    buf.seek(0); return json.loads(buf.read().decode('utf-8'))

def write_state_json(name, obj):
    body=json.dumps(obj, indent=2).encode('utf-8')
    tmp = WORKDIR / name; tmp.write_bytes(body)
    existing = drive.files().list(q=f"'{STATE_FOLDER}' in parents and name='{name}' and trashed=false",
                                  fields='files(id)').execute().get('files', [])
    media = MediaFileUpload(str(tmp), mimetype='application/json')
    if existing:
        drive.files().update(fileId=existing[0]['id'], media_body=media).execute()
    else:
        drive.files().create(body={'name':name,'parents':[STATE_FOLDER]}, media_body=media).execute()

topics_obj = read_state_json(TOPICS_NAME)
if not topics_obj or not topics_obj.get('topics'):
    topics_obj = {'topics':[{'slug':s,'description':'','example_keywords':[]} for s in SEED_TOPICS]}
    write_state_json(TOPICS_NAME, topics_obj)
TOPIC_SLUGS = [t['slug'] for t in topics_obj['topics']]
print('Seed topics:', TOPIC_SLUGS)

In [ ]:
# Step 12 — main loop: batches of BATCH_SIZE — hardened
import gc

SLUG_RE = re.compile(r'[^a-z0-9-]+')
def slugify(s): return SLUG_RE.sub('-', s.lower()).strip('-') or 'misc'

# Per-folder-per-prefix sequence cache so we don't re-list Drive every clip.
_seq_cache = {}
def next_seq(folder_id, prefix):
    key = (folder_id, prefix.lower())
    if key in _seq_cache:
        _seq_cache[key] += 1
        return _seq_cache[key]
    res = drive_retry(lambda: drive.files().list(
        q=f"'{folder_id}' in parents and trashed=false",
        fields='files(name)', pageSize=1000).execute()).get('files', [])
    pat = re.compile(rf'^{re.escape(prefix)}-(\d+)__', re.IGNORECASE)
    used=[int(m.group(1)) for f in res for m in [pat.match(f['name'])] if m]
    n = (max(used)+1) if used else 1
    _seq_cache[key] = n
    return n

report_rows = []
errors = []

for batch_start in range(0, len(pending), BATCH_SIZE):
    batch = pending[batch_start: batch_start + BATCH_SIZE]
    print(f'\n=== Batch {batch_start//BATCH_SIZE + 1} ({len(batch)} clips) ===')
    for v in batch:
        fid, fname = v['id'], v['name']
        stem = Path(fname).stem
        ext = Path(fname).suffix.lower() or '.mp4'
        print(f'  • {fname}')
        audio_path = AUDIO_DIR / f'{fid}.mp3'
        try:
            # ---------- download audio ----------
            try:
                t0=time.time(); download_audio_only(fid, audio_path); dl_t=round(time.time()-t0,1)
                no_audio_track = False
            except NoAudioTrackError:
                print('    (no audio track — routing to _review/no-audio/)')
                no_audio_track = True
                dl_t = round(time.time()-t0,1) if 't0' in dir() else 0
                tr = {'transcript':'', 'confidence':None, 'has_speech':False, 'language':None}
                cl = {'matched_topic':'general','confidence':0.0,'shot_type':'b-roll',
                      'tags':[],'reason':'no audio track'}
                tx_t = 0

            # ---------- transcribe + classify ----------
            if not no_audio_track:
                t0=time.time(); tr = transcribe_audio(audio_path); tx_t=round(time.time()-t0,1)
                try:
                    cl = classify(tr['transcript'], tr['has_speech'], TOPIC_SLUGS + ['general'])
                except Exception as e:
                    print(f'    classify-error: {e}; routing to _unmatched')
                    cl = {'matched_topic':'_unmatched','confidence':0.0,'shot_type':'b-roll',
                          'tags':[],'reason':f'classify-error: {str(e)[:200]}'}

            _safe_unlink(audio_path)

            topic = cl.get('matched_topic') or '_unmatched'
            shot  = cl.get('shot_type') or 'b-roll'
            conf  = float(cl.get('confidence') or 0.0)
            tags  = cl.get('tags', []) or []

            # ---------- route ----------
            if no_audio_track or not tr['has_speech']:
                dest_id = REVIEW_NOAUDIO; new_name = fname; status='no-audio'
            elif topic == '_unmatched' or conf < REVIEW_THRESHOLD:
                dest_id = REVIEW_UNMATCHED; new_name = fname; status='unmatched'
            else:
                shot_slug = slugify(shot if shot in ('a-roll','b-roll','talking-head') else 'b-roll')
                topic_slug = slugify(topic)
                outer = 'a-roll' if shot_slug in ('a-roll','talking-head') else 'b-roll'
                dest_id = ensure_path([outer, topic_slug], OUTPUT_ROOT)
                prefix = f'{topic_slug}-{shot_slug}'
                seq = next_seq(dest_id, prefix)
                new_name = f'{prefix}-{seq:02d}__{stem}{ext}'
                status = 'auto-placed' if conf >= AUTO_THRESHOLD else 'placed-needs-review'

            # ---------- move (re-parent only, never delete) ----------
            move_file(fid, dest_id, new_name)

            # ---------- persist state (so re-run skips this clip) ----------
            state_obj = {'id': fid, 'original_name': fname, 'new_name': new_name,
                         'topic': topic, 'shot_type': shot, 'confidence': conf,
                         'transcript': tr['transcript'], 'has_speech': tr['has_speech'],
                         'language': tr['language'], 'tags': tags,
                         'reason': cl.get('reason',''), 'status': status,
                         'timings_sec': {'download_audio': dl_t, 'transcribe': tx_t}}
            write_state_json(state_name(v), state_obj)
            report_rows.append(state_obj)
            print(f'    -> {status} | {topic}/{shot} (conf={conf}) | dl={dl_t}s tx={tx_t}s | {new_name}')
        except Exception as e:
            print(f'    ERROR (clip will retry next run): {e}')
            errors.append({'file': fname, 'id': fid, 'error': str(e)})
            _safe_unlink(audio_path)
    gc.collect()

print(f'\nDone. Processed {len(report_rows)} clips. {len(errors)} errors.')
if errors:
    print('\nErrors (will retry on next run since no state file was written):')
    for e in errors: print(' -', e['file'], '::', e['error'][:200])

In [ ]:
# Step 13 — combined transcript Google Doc (append-only; never deletes content)
# Hardcore rule honored: existing content is never overwritten or deleted.
# Tracks which file IDs are already in the doc via _state/_doc_index.json.

DOC_INDEX_NAME = '_doc_index.json'
CHUNK_CHARS = 90_000  # Docs batchUpdate per-request safety margin

def list_all_state():
    """Read every per-clip state file (skip topics.json and the doc index)."""
    out=[]; token=None
    while True:
        r=drive_retry(lambda: drive.files().list(
            q=f"'{STATE_FOLDER}' in parents and trashed=false and mimeType='application/json'",
            fields='nextPageToken, files(id,name)', pageSize=200, pageToken=token).execute())
        for f in r.get('files', []):
            if f['name'] in (TOPICS_NAME, DOC_INDEX_NAME): continue
            buf=io.BytesIO()
            dl=MediaIoBaseDownload(buf, drive.files().get_media(fileId=f['id']))
            done=False
            while not done: _, done = dl.next_chunk()
            try: out.append(json.loads(buf.getvalue().decode('utf-8')))
            except Exception: pass
        token=r.get('nextPageToken')
        if not token: break
    return out

states = list_all_state()
states.sort(key=lambda s: s.get('original_name',''))

# Find or create the transcript doc next to the raw folder.
existing = drive_retry(lambda: drive.files().list(
    q=(f"'{PARENT_ID}' in parents and name='{_q_escape(TRANSCRIPT_DOC_NAME)}' "
       f"and trashed=false and mimeType='application/vnd.google-apps.document'"),
    fields='files(id)').execute()).get('files', [])

if existing:
    doc_id = existing[0]['id']
    print(f'Reusing existing transcripts doc: {doc_id}')
else:
    try:
        created = drive_retry(lambda: drive.files().create(
            body={'name':TRANSCRIPT_DOC_NAME,'parents':[PARENT_ID],
                  'mimeType':'application/vnd.google-apps.document'},
            fields='id').execute())
        doc_id = created['id']
        print(f'Created transcripts doc: {doc_id}')
    except HttpError as e:
        raise RuntimeError(
            'Could not create transcripts Doc — Docs scope likely missing on auth.\n'
            'Fix: re-run Step 5 with force_remount=True and accept Docs scope.\n' + str(e))

# Load the index of file IDs already written to the doc.
doc_index = read_state_json(DOC_INDEX_NAME) or {'written_ids': []}
written = set(doc_index.get('written_ids') or [])

to_append = [s for s in states if s.get('id') and s['id'] not in written]
print(f'{len(states)} clips total, {len(to_append)} new to append to doc.')

if to_append:
    # Build the text to append.
    parts=[]
    for s in to_append:
        parts.append(f"{s.get('new_name') or s.get('original_name')}\n")
        parts.append(f"(was: {s.get('original_name')})  topic={s.get('topic')} "
                     f"shot={s.get('shot_type')} conf={s.get('confidence')}\n\n")
        parts.append((s.get('transcript') or '[no speech]').strip() + '\n\n')
        parts.append('-'*30 + '\n\n')
    full = ''.join(parts)

    # Append-only: insert at the very end of the doc.
    # Chunk if huge — Docs API rejects single batchUpdates above ~10MB.
    def doc_end_index():
        body = docs.documents().get(documentId=doc_id).execute()
        return body['body']['content'][-1]['endIndex']

    pos = 0
    while pos < len(full):
        chunk = full[pos:pos+CHUNK_CHARS]
        end = doc_end_index()
        insert_at = max(1, end - 1)  # before trailing newline
        docs.documents().batchUpdate(documentId=doc_id, body={'requests':[
            {'insertText':{'location':{'index': insert_at}, 'text': chunk}}
        ]}).execute()
        pos += CHUNK_CHARS

    # Update the doc-index state so we never re-append the same clip.
    doc_index['written_ids'] = list(written | {s['id'] for s in to_append})
    write_state_json(DOC_INDEX_NAME, doc_index)

print(f'\nTranscripts doc: https://docs.google.com/document/d/{doc_id}')

In [ ]:
# Step 14 — Build a CSV catalog of every processed clip
# Reads every _state/*.json, extracts the clip ID (e.g. C1932 from C1932.MP4),
# and emits a single CSV with: clip_id | topic | shot_type | confidence | status
# | summary | tags | new_filename | drive_link | transcript
#
# Output: written to Drive next to Sorted Library as "clip-catalog.csv", AND
# downloaded to Colab as /content/clip-catalog.csv (use the Files pane on the
# left to right-click → Download).

import csv, re, io
from googleapiclient.http import MediaFileUpload

CLIP_ID_RE = re.compile(r'(C\d{3,5})', re.IGNORECASE)

def extract_clip_id(name):
    """Pull C1932 etc. out of a filename. Falls back to filename stem."""
    m = CLIP_ID_RE.search(name or '')
    return m.group(1).upper() if m else (name or '').rsplit('.', 1)[0]

def drive_link(file_id):
    return f'https://drive.google.com/file/d/{file_id}/view'

def list_state_files():
    files, token = [], None
    while True:
        resp = drive_retry(lambda: drive.files().list(
            q=f"'{STATE_FOLDER}' in parents and trashed=false and name contains '.json'",
            fields='nextPageToken, files(id,name)', pageSize=1000,
            pageToken=token).execute())
        files.extend(resp.get('files', []))
        token = resp.get('nextPageToken')
        if not token: break
    return files

def read_state(file_id):
    buf = io.BytesIO()
    dl = MediaIoBaseDownload(buf, drive.files().get_media(fileId=file_id))
    done = False
    while not done: _, done = dl.next_chunk()
    buf.seek(0)
    return json.loads(buf.read().decode('utf-8'))

print('Reading state cache from Drive ...')
state_files = list_state_files()
# Skip the topics.json and doc-index — only per-clip state files
state_files = [f for f in state_files if f['name'] not in ('topics.json', '_doc_index.json')]
print(f'  {len(state_files)} clip state files found')

rows = []
for i, sf in enumerate(state_files, 1):
    try:
        s = read_state(sf['id'])
    except Exception as e:
        print(f'  [{i}/{len(state_files)}] skip {sf["name"]}: {e}')
        continue
    orig = s.get('original_name', '')
    rows.append({
        'clip_id':      extract_clip_id(orig),
        'topic':        s.get('topic', ''),
        'shot_type':    s.get('shot_type', ''),
        'confidence':   s.get('confidence', ''),
        'status':       s.get('status', ''),
        'summary':      (s.get('reason', '') or '').strip(),
        'tags':         '; '.join(s.get('tags', []) or []),
        'new_filename': s.get('new_name', ''),
        'drive_link':   drive_link(s.get('id', '')),
        'transcript':   (s.get('transcript', '') or '').strip(),
    })

# Sort by clip_id (C1932 < C1933 < ...). Numeric sort on the trailing digits.
def sort_key(r):
    m = re.search(r'(\d+)', r['clip_id'])
    return (int(m.group(1)) if m else 10**9, r['clip_id'])
rows.sort(key=sort_key)

# Write locally first
LOCAL_CSV = '/content/clip-catalog.csv'
fieldnames = ['clip_id','topic','shot_type','confidence','status','summary',
              'tags','new_filename','drive_link','transcript']
with open(LOCAL_CSV, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(rows)
print(f'Wrote {len(rows)} rows to {LOCAL_CSV}')

# Upload to Drive — replace any existing copy in the OUTPUT parent folder
CATALOG_NAME = 'clip-catalog.csv'
# Sorted Library lives inside OUTPUT_ROOT's parent. Use parent of OUTPUT_ROOT.
out_meta = drive_retry(lambda: drive.files().get(
    fileId=OUTPUT_ROOT, fields='parents,name').execute())
target_parent = out_meta['parents'][0]

existing = drive_retry(lambda: drive.files().list(
    q=f"'{target_parent}' in parents and name='{CATALOG_NAME}' and trashed=false",
    fields='files(id)').execute()).get('files', [])

media = MediaFileUpload(LOCAL_CSV, mimetype='text/csv', resumable=False)
if existing:
    new_id = drive_retry(lambda: drive.files().update(
        fileId=existing[0]['id'], media_body=media, fields='id').execute())['id']
    print(f'Updated existing  {CATALOG_NAME}  on Drive (id={new_id})')
else:
    new_id = drive_retry(lambda: drive.files().create(
        body={'name': CATALOG_NAME, 'parents': [target_parent]},
        media_body=media, fields='id').execute())['id']
    print(f'Uploaded new  {CATALOG_NAME}  to Drive (id={new_id})')

print()
print('CSV link:', f'https://drive.google.com/file/d/{new_id}/view')
print()
print('Quick stats:')
from collections import Counter
print('  by status:    ', dict(Counter(r["status"] for r in rows)))
print('  by shot_type: ', dict(Counter(r["shot_type"] for r in rows)))
print('  by topic:     ', dict(Counter(r["topic"] for r in rows)))

## Done

Your Drive is now organized:

```
<parent>/
├── <raw folder>/        ← files moved out (you can delete it later if you want)
├── Sorted Library/
│   ├── a-roll/
│   │   ├── ai-tools/
│   │   ├── e-commerce/
│   │   ├── ...auto-discovered...
│   │   └── general/
│   ├── b-roll/
│   │   ├── ai-tools/
│   │   └── general/      ← reusable cutaways
│   ├── _review/{unmatched, no-audio}/
│   └── _state/   ← cache (don't delete)
└── All Transcripts (Google Doc)
```

**Filenames keep the original at the end**, e.g. `ai-tools-a-roll-03__C2004.mp4` — so you always know which raw file became which.

**Re-running:** Re-run all cells any time you upload new clips to the raw folder. Already-processed clips are skipped via `_state/`. To force a full re-classify, delete `_state/` in Drive and re-run.